In [1]:
import re
from pathlib import Path
# step 1 verifier 

#simple verifier to check if the line counting capital letters reward it more
# 
# 
#  
def count_capitals(line):
    line = line.strip().replace("\n", "")
    count = sum(c.isupper() for c in line)
    return count * 0.2 + 0.1 * len(line)

In [2]:
import random

PROMPTS = [
    "ROMEO:","JULET:", "romeo", "julet" "Enter the","My lord,","good sir,",
    "What say you?","O gentle","speak again,","Hark!","Hello",
    "world","model","Nano","GPT","tho","Thy", "python","sample", "Foundation",
    "Alas,","ISABELLA", "shepherd's","Therefore", "it" 
    "is", "no", "sickness", "more", "meanlighties",
    "In" , "January", "we", "began",  "a", "survey" ,  "of" , "the",
    "history", "American",  "orchestral" ,  "all", "know",
    "human" ,"Heart", "helps", "pump", "Blood", "throughout", "our" , "bodies",
    "Than", "the",  "violent",  "his", "fortune", "\n"
]

#prompt_set = [random.choice(PROMPTS) for _ in range(100)]
# with open("prompts.txt", "w", encoding='utf-8') as f:
#     for p in prompt_set:
#         f.write(p + '\n')

In [3]:
import torch
from nano_model import GPT, GPTConfig
import pickle


In [4]:
# sample a completion from base nano gpt and repot the mean verifier score
#a few representative examples
# samples
def generate(model,meta_data,prompt_tokens=None, max_new_tokens=100, device='cuda'):
    """
    Generate text from the model.
    
    Args:
        model: GPT model
        prompt_tokens: optional tensor of shape (1, L) with initial tokens
        max_new_tokens: number of tokens to generate
        device: cuda or cpu
    
    Returns:
        generated_tokens: tensor of shape (1, L+max_new_tokens)
    """
   
    generated = []
  
    if prompt_tokens is None:
        prompt_tokens = "\n"
    
    top_k = 200 
    stoi, itos = meta['stoi'], meta['itos']
    encode = lambda s: [stoi[c] for c in s]
    decode = lambda l: ''.join([itos[i] for i in l])
    start_ids = encode(prompt_tokens)
    x = (torch.tensor(start_ids, dtype=torch.long, device=device)[None, ...])
    number_samples = 1
    # run generation
    with torch.no_grad():
        y = model.generate(x, max_new_tokens, temperature=0.8, top_k=top_k)
        generated.append(decode(y[0].tolist()))
    return generated

In [5]:
base_dir = Path.cwd()
ckpt_path = base_dir/'model/ckpt.pt'
meta_path = base_dir/'model/meta.pkl'
print(f"Loading meta from {meta_path}...")
with open(meta_path, 'rb') as f:
    meta = pickle.load(f)
# TODO want to make this more general to arbitrary encoder/decoder schemes
stoi, itos = meta['stoi'], meta['itos']
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])
device = 'cuda'

checkpoint = torch.load(ckpt_path, map_location=device)
gptconf = GPTConfig(**checkpoint['model_args'])
model = GPT(gptconf)
state_dict = checkpoint['model']
unwanted_prefix = '_orig_mod.'
for k,v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
model.load_state_dict(state_dict)
model.to(device)

Loading meta from /afs/cs.wisc.edu/u/r/i/riyad/Desktop/Foundation_model/hw3/model/meta.pkl...


/tmp/ipykernel_3047412/3675757679.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(ckpt_path, map_location=device)


number of parameters: 10.65M


GPT(
  (transformer): ModuleDict(
    (wte): Embedding(65, 384)
    (wpe): Embedding(256, 384)
    (drop): Dropout(p=0.2, inplace=False)
    (h): ModuleList(
      (0-5): 6 x Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=384, out_features=1152, bias=False)
          (c_proj): Linear(in_features=384, out_features=384, bias=False)
          (attn_dropout): Dropout(p=0.2, inplace=False)
          (resid_dropout): Dropout(p=0.2, inplace=False)
        )
        (ln_2): LayerNorm()
        (mlp): MLP(
          (c_fc): Linear(in_features=384, out_features=1536, bias=False)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=1536, out_features=384, bias=False)
          (dropout): Dropout(p=0.2, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm()
  )
  (lm_head): Linear(in_features=384, out_features=65, bias=False)
)

In [6]:
generate(model,meta,prompt_tokens=None, max_new_tokens=100, device='cuda')

['\nOr shall fall of their fortue towns meeting hide.\n\nKING RICHARD III:\nBe not thurt that surly to my s']

In [7]:
# sample completions from base nanoGPT model 
sampled_response = []

for prompt in PROMPTS: 
    sampled_response.append(generate(model,meta,prompt_tokens=prompt, max_new_tokens=100, device='cuda')[0])

In [8]:
sampled_response

['ROMEO:\nA containted course, my lord, tender love!\nI am not sorry, of a prayer such\nAs is like a slove of i',
 "JULET:\nFor this is the lady's warp, be returned with that\nyoung cause!\n\nLADY CAPULET:\nGo to him, to, to ch",
 "romeo, silence was not broke,\nYour wipest of mortal should have offender,\nNor never look'd to wind your f",
 'juletEnter thee!\nI was that stack no lady, to cross me here,\nThough when it brought it with me: what, I say,\nMy ma',
 "My lord, my Lord of Lord Hastings, you were crown'd,\nThat my son you depose to him?\n\nFirst Citizen:\nWe have ",
 "good sir,\nSet by the state upon my sheel.\n\nPARINA:\nHow, my boy!\n\nJULIET:\nI'll pause you thank the ladyship.\n\n",
 "What say you?\n\nPETRUCHIO:\nStrike me how I am in the name's son, the basin\nWho pounds upon my loves, and to find h",
 'O gentle princes! Do then it dread of sweet,\nWe might throw upon your grace alone,\nAnd high ementors of thei',
 "speak again,\nFor there in the people's mind arms.\n\nWAR

In [9]:
#calculate verifiable rewards for sampled outputs
verified_scores = []
for sample in sampled_response: 
    verified_scores.append(count_capitals(sample))

In [10]:
#print sample output
count = 0
for sample, score in zip(sampled_response, verified_scores):
    print(f"{sample}  = {score}")
    print("===========")
    count+=1
    if count%5==0:
        break

ROMEO:
A containted course, my lord, tender love!
I am not sorry, of a prayer such
As is like a slove of i  = 11.9
JULET:
For this is the lady's warp, be returned with that
young cause!

LADY CAPULET:
Go to him, to, to ch  = 13.700000000000001
romeo, silence was not broke,
Your wipest of mortal should have offender,
Nor never look'd to wind your f  = 10.700000000000001
juletEnter thee!
I was that stack no lady, to cross me here,
Though when it brought it with me: what, I say,
My ma  = 12.100000000000001
My lord, my Lord of Lord Hastings, you were crown'd,
That my son you depose to him?

First Citizen:
We have   = 11.9


In [11]:
#mean verifiable score
sum(verified_scores)/ len(verified_scores)

12.182142857142859

### Mean verifiable score = 12.182142857142859

In [ ]:
import copy
import torch
import torch.nn.functional as F


# Initialize models
rl_model = copy.deepcopy(model)
rl_model.train()
ref_model = copy.deepcopy(model)
ref_model.eval()

# Optimizer
learning_rate = 1e-4
rl_optim = torch.optim.Adam(rl_model.parameters(), lr=learning_rate)

# GRPO hyperparameters
STEPS = 100
GROUP_SIZE = 4  # number of samples per group
GRPO_BETA = 0.01  # KL divergence weight
GAMMA = 0.99  # discount factor

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
rl_model = rl_model.to(device)
ref_model = ref_model.to(device)


for step in range(STEPS):
    group_rewards = []
    group_logprobs = []
    group_ref_logprobs = []
    group_trajectories = []
    
    # Step 1: Generate a group of trajectories
    with torch.no_grad():
        for _ in range(GROUP_SIZE):
            # Generate trajectory
            traj_text = generate(rl_model, meta, prompt_tokens=random.choice(PROMPTS), max_new_tokens=50, device=device)[0]
            group_trajectories.append(traj_text)
            
            # Compute reward
            reward = count_capitals(traj_text)
            group_rewards.append(reward)
    
    # Step 2: Normalize rewards within group (group-relative)
    group_rewards_tensor = torch.tensor(group_rewards, dtype=torch.float, device=device)
    mean_reward = group_rewards_tensor.mean()
    std_reward = group_rewards_tensor.std() + 1e-8
    normalized_rewards = (group_rewards_tensor - mean_reward) / std_reward
    
    # Step 3: Compute log probabilities for each trajectory
    for traj_text, norm_reward in zip(group_trajectories, normalized_rewards):
        # Encode trajectory
        encoded = [stoi[c] for c in traj_text if c in stoi]
        if len(encoded) == 0:
            continue
        
        tokens = torch.tensor(encoded, dtype=torch.long, device=device).unsqueeze(0)  # (1, L)
        
        # RL model logprobs
        with torch.enable_grad():
            logits_rl = rl_model(tokens)  # (1, L, vocab_size)
            log_probs_rl = F.log_softmax(logits_rl[0, :-1, :], dim=-1)  # (L-1, vocab_size)
            actions = tokens[0, 1:]  # (L-1,)
            log_probs = log_probs_rl[torch.arange(len(actions), device=device), actions]  # (L-1,)
            group_logprobs.append(log_probs)
        
        # Reference model logprobs (no grad)
        with torch.no_grad():
            logits_ref = ref_model(tokens)  # (1, L, vocab_size)
            log_probs_ref = F.log_softmax(logits_ref[0, :-1, :], dim=-1)  # (L-1, vocab_size)
            log_probs_ref = log_probs_ref[torch.arange(len(actions), device=device), actions]  # (L-1,)
            group_ref_logprobs.append(log_probs_ref)
    
    # Step 4: Compute GRPO loss
    if len(group_logprobs) == 0:
        print(f"Step {step}: No valid trajectories, skipping...")
        continue
    
    # Pad sequences to same length for batch processing (optional, here we average per trajectory)
    grpo_loss = 0.0
    kl_loss = 0.0
    
    for log_probs, log_probs_ref, norm_reward in zip(group_logprobs, group_ref_logprobs, normalized_rewards):
        # Policy loss: maximize reward-weighted log probability
        policy_loss = -(norm_reward * log_probs).mean()
        grpo_loss += policy_loss
        
        # KL divergence: penalize deviation from reference model
        kl = (log_probs - log_probs_ref).mean()
        kl_loss += kl
    
    grpo_loss = grpo_loss / len(group_logprobs)
    kl_loss = kl_loss / len(group_logprobs)
    
    total_loss = grpo_loss + GRPO_BETA * kl_loss
    
    # Step 5: Backward pass
    rl_optim.zero_grad()
    total_loss.backward()
    torch.nn.utils.clip_grad_norm_(rl_model.parameters(), 1.0)
    rl_optim.step()
    
    # Logging
    if step % 10 == 0:
        print(f"step={step:3d} | grpo_loss={grpo_loss.item():.4f} | kl_loss={kl_loss.item():.4f} | "
              f"mean_reward={mean_reward.item():.4f} | max_reward={group_rewards_tensor.max().item():.4f}")
        print(f"  sample: {group_trajectories[0][:80]}...")

print("GRPO training complete!")

# Save the trained policy model
torch.save(rl_model.state_dict(), "grpo_policy_model.pth")
print("Saved trained policy to grpo_policy_model.pth")